In [3]:
#@title 1. Setup Environment
!pip install -q "transformers>=4.45.0,<5.0.0" accelerate einops pillow pandas

import os
from google.colab import drive

# # Mount Google Drive to access your image folder
# drive.mount('/content/drive')

print("✅ Setup complete. Please proceed to the next cell.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
✅ Setup complete. Please proceed to the next cell.


In [ ]:
#@title 2. Resumable Detailed Captioning { display-mode: "form" }
#@markdown ### ⚙️ Configuration
#@markdown Enter the path to your image folder in Google Drive, then click **Run**.

FOLDER_PATH = "/content/drive/MyDrive/vivo办公套件" #@param {type:"string"}

MODEL_CHOICE = "Moondream 2B (recommended)" #@param ["Moondream 2B (recommended)", "Moondream 0.5B (fastest)"]

CAPTION_PROMPT = "Describe this image in detail." #@param {type:"string"}

# ================= SCRIPT (do not edit below) =================
import os
import pandas as pd
from PIL import Image
import torch
from transformers import AutoModelForCausalLM
from tqdm.auto import tqdm
import time

# Map GUI choice to actual Hugging Face model IDs
MODEL_MAP = {
    "Moondream 2B (recommended)": {
        "id": "vikhyatk/moondream2",
        "revision": "2025-06-21",
    },
    "Moondream 0.5B (fastest)": {
        "id": "vikhyatk/moondream2",
        "revision": "2024-08-26",  # 0.5B era revision (smaller & faster)
    },
}

MODEL_ID = MODEL_MAP[MODEL_CHOICE]["id"]
MODEL_REVISION = MODEL_MAP[MODEL_CHOICE]["revision"]

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}


def get_image_files(folder_path):
    """Recursively finds all image files in a folder and its subfolders."""
    image_files = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if os.path.splitext(file)[1].lower() in IMAGE_EXTENSIONS:
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, folder_path)
                image_files.append(rel_path)
    return sorted(image_files)


def load_model():
    """Loads the Moondream model onto the GPU."""
    print(f"🔄 Loading model: {MODEL_ID} @ {MODEL_REVISION}")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        trust_remote_code=True,
        device_map={"": "cuda"},
        torch_dtype=torch.float16,
    )
    print("✅ Model loaded.")
    return model


def main():
    print(f"📂 Folder: {FOLDER_PATH}")
    print(f"🤖 Model : {MODEL_CHOICE}")
    print(f"📝 Prompt: {CAPTION_PROMPT}\n")

    if not os.path.isdir(FOLDER_PATH):
        print(f"❌ Error: The folder '{FOLDER_PATH}' does not exist.")
        return

    image_files = get_image_files(FOLDER_PATH)
    if not image_files:
        print(f"⚠️ No supported images found in '{FOLDER_PATH}'.")
        return
    print(f"📂 Found {len(image_files)} image(s).")

    csv_path = os.path.join(FOLDER_PATH, "descriptions.csv")

    if os.path.exists(csv_path):
        print(f"📄 Found existing CSV: {csv_path}. Resuming...")
        df = pd.read_csv(csv_path)
        if "filename" not in df.columns or "description" not in df.columns:
            print("❌ CSV missing 'filename' or 'description' columns.")
            return
        existing_files = set(df["filename"].tolist())
        new_files = [f for f in image_files if f not in existing_files]
        if new_files:
            print(f"➕ Adding {len(new_files)} new file(s) to CSV.")
            new_rows = pd.DataFrame({"filename": new_files, "description": ""})
            df = pd.concat([df, new_rows], ignore_index=True)
            df.to_csv(csv_path, index=False)
    else:
        print(f"🆕 Creating new CSV: {csv_path}")
        df = pd.DataFrame({"filename": image_files, "description": ""})
        df.to_csv(csv_path, index=False)

    df["description"] = df["description"].fillna("").astype(str).str.strip()
    pending_mask = df["description"] == ""
    pending_files = df[pending_mask]["filename"].tolist()

    if not pending_files:
        print("🎉 All images already have descriptions. Nothing to do.")
        return

    print(f"⏳ {len(pending_files)} image(s) still need descriptions.\n")

    model = load_model()

    print("🚀 Starting caption generation...")
    for rel_path in tqdm(pending_files, desc="Captioning"):
        full_path = os.path.join(FOLDER_PATH, rel_path)
        try:
            image = Image.open(full_path).convert("RGB")
            result = model.query(image, CAPTION_PROMPT)
            description = result["answer"].strip()
            if not description:
                description = "No description generated."
        except Exception as e:
            description = f"ERROR: {e}"

        df.loc[df["filename"] == rel_path, "description"] = description
        df.to_csv(csv_path, index=False)  # Save after EACH image (resumable)
        time.sleep(0.1)

    print(f"\n✅ All done! Results saved to: {csv_path}")


if __name__ == "__main__":
    main()

📂 Folder: /content/drive/MyDrive/vivo办公套件
🤖 Model : Moondream 2B (recommended)
📝 Prompt: Describe this image in detail.

📂 Found 198 image(s).
📄 Found existing CSV: /content/drive/MyDrive/vivo办公套件/descriptions.csv. Resuming...
⏳ 198 image(s) still need descriptions.

🔄 Loading model: vikhyatk/moondream2 @ 2025-06-21


`torch_dtype` is deprecated! Use `dtype` instead!


generation_config.json:   0%|          | 0.00/69.0 [00:00<?, ?B/s]

✅ Model loaded.
🚀 Starting caption generation...


Captioning:   0%|          | 0/198 [00:00<?, ?it/s]